# Entraînement du CNN — détection de pneumonie (NORMAL / PNEUMONIA)

Dataset : *Chest X-Ray Images (Pneumonia)* (Paul Mooney, Kaggle), 5 863 images, 2 classes.

Ce notebook a produit `weights/cnn_pneumonia_model.pth`, utilisé par `src/vision_module.py`. Résultats de cette exécution sur le jeu de test : accuracy 77,24 %, matrice de confusion `[[97, 137], [5, 385]]`. Aucune graine aléatoire n'était fixée : les résultats varient légèrement d'une exécution à l'autre (voir `src/train_vision.py` pour une version reproductible).

*Note : la sortie de la cellule de décompression (`unzip`) a été supprimée pour alléger le fichier.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip "/content/drive/MyDrive/archive.zip" -d "/content/chest_xray"

In [ ]:
import os
os.listdir("/content/chest_xray")

['chest_xray']

In [ ]:
import os

base_dir = "/content/chest_xray/chest_xray"
print(os.listdir(base_dir))
print(os.listdir(base_dir + "/train"))

['test', 'train', '__MACOSX', 'chest_xray', 'val']
['NORMAL', 'PNEUMONIA']


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

import numpy as np

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


Transformations images

In [ ]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [ ]:
train_dataset = datasets.ImageFolder(
    root="/content/chest_xray/chest_xray/train",
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root="/content/chest_xray/chest_xray/test",
    transform=transform
)

val_dataset = datasets.ImageFolder(
    root="/content/chest_xray/chest_xray/val",
    transform=transform
)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
print(train_dataset.classes)

['NORMAL', 'PNEUMONIA']


In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

In [ ]:
model = CNN().to(device)

class_weights = torch.tensor([2.0, 1.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.0005)

In [ ]:
epochs = 8

train_losses = []

for epoch in range(epochs):

    model.train()
    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    train_losses.append(epoch_loss)

    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}")

Epoch 1/8 - Loss: 0.2613
Epoch 2/8 - Loss: 0.1244
Epoch 3/8 - Loss: 0.0942
Epoch 4/8 - Loss: 0.0727
Epoch 5/8 - Loss: 0.0600
Epoch 6/8 - Loss: 0.0574
Epoch 7/8 - Loss: 0.0491
Epoch 8/8 - Loss: 0.0352


In [ ]:
model.eval()

correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 77.24%


In [ ]:
cm = confusion_matrix(all_labels, all_preds)

print(cm)

[[ 97 137]
 [  5 385]]


In [ ]:
torch.save(model.state_dict(), "cnn_pneumonia_model.pth")

In [ ]:
from google.colab import files
files.download("cnn_pneumonia_model.pth")